In [1]:
from openai import OpenAI
# openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [2]:
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Did you know that there is a species of jellyfish that is immortal?! The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage and grow back into an adult again, making it theoretically immortal! Isn\'t that mind-blowing?'

In [16]:
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Do you know the shortcut to switch between tabs in cursor app"}])

response.choices[0].message.content

"I'm not aware of any specific shortcuts for switching between tabs in the Cursor app. However, I can suggest some general methods that may work on various Android devices:\n\n1. Swipe gesture: You can try swiping left or right on the tab icon with your finger to switch between tabs.\n2. Gestures from the bottom: Many modern smartphones support gestures from the bottom of the screen. To navigate through tabs, you might want to swipe down and use two fingers to tap on the next/previous tab.\n\nPlease let me know if any of these methods are helpful or would you like more assistance"

In [3]:
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [4]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant trashes websites after analyzeing the contents of a website \
and provides a hillarious short summary"

In [5]:
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [6]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [7]:
# A class to represent a Webpage
# If you're not familiar with Classes, check out the "Intermediate Python" notebook

# Some websites need you to use proper headers when fetching them:
# Import necessary modules
# selenium is in pyproject.toml; run 'uv sync' in project root if missing
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

class ScrapeWebsite:
    def __init__(self, url):
        """
        Create this Website object from the given URL using Selenium + BeautifulSoup
        Supports JavaScript-heavy and normal websites uniformly.
        """
        self.url = url
        driver = None
        
        try:
            # Configure headless Chrome with better options for JS-heavy sites
            options = Options()
            options.add_argument('--headless')
            options.add_argument('--no-sandbox')
            options.add_argument('--disable-dev-shm-usage')
            options.add_argument('--disable-gpu')
            options.add_argument('--window-size=1920,1080')
            # Add user agent to avoid bot detection
            options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
            # Disable automation flags
            options.add_experimental_option("excludeSwitches", ["enable-automation"])
            options.add_experimental_option('useAutomationExtension', False)

            # Use webdriver-manager to manage ChromeDriver
            service = Service(ChromeDriverManager().install())

            # Initialize the Chrome WebDriver with the service and options
            driver = webdriver.Chrome(service=service, options=options)
            
            # Execute script to hide webdriver property
            driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
                'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
            })

            # Start Selenium WebDriver
            print(f"Loading {url}...")
            driver.get(url)

            # Wait for page to load - use WebDriverWait instead of just time.sleep
            # Wait for body element to be present (indicates page has loaded)
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            # Additional wait for JavaScript to execute
            time.sleep(3)
            
            # Scroll down to trigger lazy-loaded content
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            # Scroll back up
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(1)

            # Fetch the page source after JS execution
            page_source = driver.page_source
            
            # Parse the HTML content with BeautifulSoup
            soup = BeautifulSoup(page_source, 'html.parser')

            # Extract title
            self.title = soup.title.string if soup.title else "No title found"

            # Remove unnecessary elements
            if soup.body:
                for irrelevant in soup.body(["script", "style", "img", "input", "noscript"]):
                    irrelevant.decompose()
                self.text = soup.body.get_text(separator="\n", strip=True)
            else:
                self.text = ""
                
            print(f"Successfully scraped: {self.title[:50]}...")
            
        except Exception as e:
            print(f"Error scraping {url}: {e}")
            self.title = "Error loading page"
            self.text = f"Failed to load content: {str(e)}"
        finally:
            if driver:
                driver.quit()


In [8]:
def summarize_js_website(url):
    website = ScrapeWebsite(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [9]:
summary = summarize_js_website("https://www.thehindu.com/news/national/union-cabinet-clears-urban-challenge-fund-for-cities/article70631961.ece")

Loading https://www.thehindu.com/news/national/union-cabinet-clears-urban-challenge-fund-for-cities/article70631961.ece...
Successfully scraped: 
	Union Cabinet clears ‘Urban Challenge Fund’ for ...


In [10]:
display(Markdown(summary))

**The Union Cabinet Has Funded the Urban Challenge Fund, Which Could Potentially Be A Game Changer In Indian Cities**
=================================================================================================
### The New Scheme:
Union Cabinet clears ‘Urban Challenge Fund’ for cities, a new Centrally sponsored scheme of the Ministry of Housing and Urban Affairs.

### 25% Government Subsidy:  
The scheme aims to create a competitive tendering process where cities will compete for projects that are bankable enough to attract market funding of at least 50% from other sources. The amount allocated will range between ₹1 lakh crore to ₹4 lakh crore, depending on the projects submitted.

### Benefits In Focus
This could be beneficial since there are various aspects that need fixing in the urban area, which include transportation systems, housing shortage, water and sanitation facilities etc. A major push from a single scheme could bring about positive changes which could turn Indian cities into high-growth hubs.
================================================================================================

### Key Details 
#### Union Minister of Housing and Urban Affairs, and Power, Manohar Lal, will chair a budget review meeting to implement the UCF.


*   The fund is expected to be operational from FY 2025-26 to FY 2030-31, with an extendable implementation period up to FY 2033-34.

#### Funding Allocation:
The Central assistance will cover 25% of project costs subject to reaching a minimum 50% through contributions from the market. The total expected investment in the urban sector is ₹4 lakh crore over the next five years.

#### Eligibility Criteria: 
*   All cities with a population of 10 lakhs or more (based on estimates)
*   Capitals of States and Union Territories not already included
*   Major industrial cities with a population of 1 lac or more
*   Smaller urban local bodies in hilly states, northeastern states, and below 1-lac population

#### Additional Scheme 
There’s also another credit repayment guarantee scheme available for smaller ones.

#### What This Means For India:
Indian cities can focus on sustainability when creating growth using the aid from this new fund from central authorities.